# LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a terminal-style launcher. Notebook cells are shell commands; clone/pull, Drive binding, and execution logic live in `scripts/*.py` and `src/`.


## Operating Model

1. Run the Drive mount code cell. This is a Colab platform exception, not experiment logic.
2. Run the GCS auth code cell if you want to read the interactive pilot source directly from `gs://.../uncompressed/tf_example/...`.
3. Bootstrap the repository and bind artifacts to Drive-backed storage.
4. Stage the fixed 10-shard `validation_interactive` tf_example pilot subset into the Drive-bound `assets/raw_womd` cache.
5. Preprocess that staged pilot subset once, then archive it once.
6. In future Colab sessions, restore the pilot archive before running pretrained model evaluations.


## Mount Google Drive

Mount Drive once per Colab runtime so checkpoints, preprocessed artifacts, run bundles, and debug outputs can bind to the persistent project folder.


In [ ]:
from pathlib import Path

DRIVE_MOUNTPOINT = "/content/drive"
drive_root = Path(DRIVE_MOUNTPOINT) / "MyDrive"

if drive_root.is_dir():
    print(f"Drive already mounted at {drive_root}")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNTPOINT)
    print(f"Mounted Drive at {drive_root}")


## Authenticate GCS Access

Authenticate the Colab runtime for direct `gs://waymo_open_dataset_motion_v_1_1_0` reads. Keep this enabled unless you intentionally switch to the Drive-local raw WOMD cache.


In [ ]:
from __future__ import annotations

import json
import subprocess

USE_GCS_DIRECT = True

if USE_GCS_DIRECT:
    from google.colab import auth
    auth.authenticate_user()
    checks = {}
    commands = {
        "active_account": ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=json"],
        "adc_token": ["gcloud", "auth", "application-default", "print-access-token"],
    }
    for name, command in commands.items():
        proc = subprocess.run(command, text=True, capture_output=True, check=False)
        payload = {"command": command, "returncode": proc.returncode}
        if name == "adc_token":
            payload["token_ready"] = proc.returncode == 0 and bool(proc.stdout.strip())
            payload["stderr"] = proc.stderr.strip()
        else:
            payload["stdout"] = proc.stdout.strip()
            payload["stderr"] = proc.stderr.strip()
        checks[name] = payload
    print(json.dumps({"use_gcs_direct": True, "checks": checks}, indent=2))
else:
    print(json.dumps({
        "use_gcs_direct": False,
        "message": "Skipping GCS auth because this session will use the Drive-local assets/raw_womd cache.",
    }, indent=2))


## Bootstrap Repository And Bind Artifacts

Clone or fast-forward `main`, validate the Drive mount, and bind the repository artifact paths to persistent Drive-backed storage.


In [ ]:
%%bash
set -euo pipefail

BOOTSTRAP=/tmp/latentdriver_colab_bootstrap.py
curl -fsSL \
  https://raw.githubusercontent.com/achyutmorang/latentdriver-waymax-experiments/main/scripts/colab_bootstrap.py \
  -o "$BOOTSTRAP"

python3 "$BOOTSTRAP" \
  --repo-url https://github.com/achyutmorang/latentdriver-waymax-experiments.git \
  --branch main \
  --repo-dir /content/latentdriver-waymax-experiments \
  --drive-base-root /content/drive/MyDrive/waymax_research \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


## List Available Runner Profiles

Print the supported `colab_canary.py` profiles. Use this when you need to confirm the exact profile names before launching a run.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
python3 scripts/colab_canary.py --list-profiles


## Confirm Git Revision

Fast-forward the local Colab checkout and print the active commit. This is a lightweight sanity check before expensive runs.


In [ ]:
%%bash
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main
git rev-parse --short HEAD


## Stage Fixed 10-Shard `validation_interactive` Pilot

Point the runner at the tf_example interactive validation split, then materialize the fixed dense 10-shard pilot subset into `artifacts/assets/raw_womd/validation_interactive_pilot/...`. The source must be the `tf_example` path, not `scenario`, because the current LatentDriver/Waymax stack parses tf.Example records.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# Keep this pointed at the tf_example interactive validation split.
export LATENTDRIVER_INTERACTIVE_PILOT_SOURCE_URI="gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/tf_example/validation_interactive/validation_interactive_tfexample.tfrecord@150"

python3 scripts/colab_canary.py --profile stage-interactive-pilot-shards


## Preprocess The Staged Interactive Pilot Cache

This builds the dedicated pilot preprocess cache under `artifacts/assets/preprocessed/interactive_pilot/...`. Run the status profile before and after the preprocess profile so you can confirm the cache is complete before archiving it.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-status
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess --auto-install-runtime
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-status


## Create Stable Archive For Future Pilot Evaluations

Once the 10-shard pilot preprocess cache is complete, archive it to Drive-backed storage. This gives you one persistent tar file that future Colab runtimes can restore before evaluating `IDM`, `LatentDriver`, `PlanT`, or your modulated variant on the same fixed pilot subset.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-archive-status
python3 scripts/colab_canary.py --profile create-interactive-pilot-preprocess-archive
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-archive-status


## Track Interactive Pilot Archive Progress From Colab Terminal

If the archive cell is running, paste this into the Colab terminal to watch the temporary tar file grow, inspect the latest archive log, and confirm when the final archive appears.

```bash
cd /content/latentdriver-waymax-experiments && while true; do clear; date -u; echo; echo "== active archive processes =="; ps -eo pid,etime,cmd | grep -E "interactive_pilot|preprocess_cache_archive|tar -C" | grep -v grep || echo "no archive process found"; echo; echo "== pilot archive files =="; ls -lh artifacts/assets/preprocessed/.interactive_pilot_preprocess_cache.tar.tmp artifacts/assets/preprocessed/interactive_pilot_preprocess_cache.tar 2>/dev/null || echo "archive not created yet"; echo; echo "== latest archive stdout =="; LATEST=$(ls -td results/debug_runs/*create_interactive_pilot_preprocess_archive 2>/dev/null | head -n 1); echo "${LATEST:-no archive debug run yet}"; [ -n "${LATEST:-}" ] && tail -n 80 "$LATEST/steps/01_create_interactive_pilot_preprocess_archive.stdout.log" 2>/dev/null; sleep 60; done
```


## Restore The Pilot Archive In Future Colab Sessions

In later Colab runs, restore the archived interactive pilot cache before launching pretrained model evaluations. This should be the first cache step in any fresh runtime after Drive mount, GCS auth, and repository bootstrap.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-archive-status
python3 scripts/colab_canary.py --profile restore-interactive-pilot-preprocess-archive
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-archive-status


## Later: Run A Full Reactive Simulation

This remains a separate heavier workflow. Use it only after the interactive pilot preprocessing and archive flow is stable and you intentionally want the full validation evaluation path.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# Next milestone: one resumable full reactive simulation/evaluation run for LatentDriver T2-J3.
# If Colab disconnects, rerun this same cell; completed shards are skipped.
python3 scripts/colab_canary.py \
  --profile full-eval-reactive-single \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --auto-install-runtime \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


## Inspect Latest Debug Bundle

Use this after a failed canary run. It prints the latest run manifest and latest failure summary without requiring manual traceback hunting.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

# Use this after any failed canary run.
python3 - <<'PY'
import json
from pathlib import Path

latest = Path("results/debug_runs/latest/manifest.json")
latest_failure = Path("results/debug_runs/latest_failure/failure_summary.json")
for path in (latest, latest_failure):
    print(f"\n== {path} ==")
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2)[:12000])
    else:
        print("missing")
PY
